In [8]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

In [6]:

# ─────────────────────────────────────────
# LOAD ALL RIDER PICKUP LOCATIONS
# ─────────────────────────────────────────

def parse_coord(s):
    s = str(s).strip('()').split(',')
    return float(s[0]), float(s[1])

riders = pd.read_excel('riders.xlsx')

# Use ALL riders — both statuses show real demand locations
all_riders = riders.copy()
all_riders[['px','py']] = pd.DataFrame(
    all_riders['pickup_location'].apply(parse_coord).tolist(),
    index=all_riders.index)

X = all_riders[['px','py']].values

print(f"Total riders        : {len(riders)}")
print(f"  dropped-off       : {(riders['status']=='dropped-off').sum()}")
print(f"  abandoned         : {(riders['status']=='abandoned').sum()}")
print(f"  other             : {(~riders['status'].isin(['dropped-off','abandoned'])).sum()}")
print(f"Using ALL {len(X)} pickup points for KMeans\n")


Total riders        : 34421
  dropped-off       : 34117
  abandoned         : 279
  other             : 25
Using ALL 34421 pickup points for KMeans



In [7]:

# ─────────────────────────────────────────
# FIT KMEANS FOR K = 1 TO 10
# ─────────────────────────────────────────

print("=" * 60)
print("  HUB LOCATIONS FROM KMEANS (K=1 to K=10)")
print("  All coordinates fitted on real BoxCar pickup data")
print("=" * 60)

all_hubs = {}

for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X)

    # Sort centres by cluster size (largest demand zone first)
    sizes   = np.bincount(km.labels_)
    order   = np.argsort(-sizes)   # descending by size
    centres = km.cluster_centers_[order]
    sizes   = sizes[order]

    all_hubs[k] = centres

    print(f"\n# K={k} hub{'s' if k>1 else ''}")
    print(f"HUBS = [")
    for i, (c, s) in enumerate(zip(centres, sizes)):
        pct = s / len(X) * 100
        print(f"    np.array([{c[0]:6.2f}, {c[1]:6.2f}]),   "
              f"# Hub {i+1}  —  {s:,} riders ({pct:.1f}%)")
    print(f"]")

print("\n" + "=" * 60)

  HUB LOCATIONS FROM KMEANS (K=1 to K=10)
  All coordinates fitted on real BoxCar pickup data

# K=1 hub
HUBS = [
    np.array([  8.36,  12.32]),   # Hub 1  —  34,421 riders (100.0%)
]

# K=2 hubs
HUBS = [
    np.array([  5.51,  10.34]),   # Hub 1  —  18,017 riders (52.3%)
    np.array([ 11.50,  14.51]),   # Hub 2  —  16,404 riders (47.7%)
]

# K=3 hubs
HUBS = [
    np.array([  5.43,  14.86]),   # Hub 1  —  11,993 riders (34.8%)
    np.array([ 12.84,  14.10]),   # Hub 2  —  11,586 riders (33.7%)
    np.array([  6.78,   7.60]),   # Hub 3  —  10,842 riders (31.5%)
]

# K=4 hubs
HUBS = [
    np.array([  5.61,  15.00]),   # Hub 1  —  11,013 riders (32.0%)
    np.array([ 12.83,  15.84]),   # Hub 2  —  8,143 riders (23.7%)
    np.array([ 11.36,   9.08]),   # Hub 3  —  7,638 riders (22.2%)
    np.array([  4.54,   7.88]),   # Hub 4  —  7,627 riders (22.2%)
]

# K=5 hubs
HUBS = [
    np.array([  4.13,  13.84]),   # Hub 1  —  8,062 riders (23.4%)
    np.array([  9.03,  16.16]),   # Hub 2  —  7,9